# Lecture 5.6: Deterministic Pipelines, Chaining Agents in Code

**Section 05: Multi-Agent Orchestration & Guardrails**

In this notebook you will build multi-agent pipelines where **your Python code**, not the model, decides what happens next. You'll implement three code-driven orchestration patterns from the OpenAI Agents SDK: a structured-output gate, a sequential agent chain, and a while-loop with an evaluator agent.

## Cell 1: Install the OpenAI Agents SDK

This notebook uses the **OpenAI Agents SDK**, the Python framework this course is built on. The cell below installs it with `pip`.

The version is pinned to a specific release so that the examples in this notebook behave exactly as shown, regardless of what has changed in newer releases of the SDK. If the package is already installed in this Colab session (for example, because you ran an earlier cell in the same session), this command will simply confirm the pinned version is present and complete almost instantly.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00


## Cell 2: Configure Your OpenAI API Key

Agents in this notebook call the OpenAI API, so the SDK needs your API key available as the `OPENAI_API_KEY` environment variable.

**In Google Colab:**
1. Click the key icon (🔑) in the left sidebar to open **Secrets**.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.
5. Run the cell below. It reads the secret and writes it into the environment for this session.

**If you are running locally instead of in Colab**, skip the Colab Secrets step and set the environment variable in your terminal before launching Jupyter, for example: `export OPENAI_API_KEY="sk-..."`. Do not hardcode your key in the notebook.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the Model Name

Every agent in this notebook references a single `MODEL_NAME` variable instead of a hardcoded model string. Changing this one variable updates the model used by every agent you define below, which makes it easy to try a different model across the whole notebook.

The comment links to OpenAI's models page so you can check what is currently available before you pick a model for your own projects.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

| Import | Used for |
|---|---|
| `dataclass` | Defining `EvaluationFeedback`, the evaluator's structured output type |
| `Literal` | Restricting the evaluator's `score` field to a fixed set of string values |
| `BaseModel` (pydantic) | Defining `OutlineCheckerOutput`, the quality-gate's structured output type |
| `Reasoning` | Configuring reasoning effort on `ModelSettings` |
| `Agent` | Defining each agent in the pipeline |
| `ItemHelpers` | Extracting plain text from an agent's message output items |
| `ModelSettings` | Tuning reasoning effort and verbosity per agent |
| `RunConfig` | Linking every `Runner.run()` call in a pipeline together for tracing |
| `Runner` | Executing each agent |
| `TResponseInputItem` | Typing the growing input list used in the evaluator loop |

One import is new compared to earlier lectures in this section: `RunConfig`. In Lecture 4.4 you used `RunConfig` to set `max_turns` and `workflow_name` for a single run. Here you'll use its `group_id` field to link several separate `Runner.run()` calls together in the traces dashboard, so a whole pipeline shows up as one connected flow instead of unrelated runs.

In [4]:
from dataclasses import dataclass
from typing import Literal

from pydantic import BaseModel
from openai.types.shared import Reasoning

from agents import (
    Agent,
    ItemHelpers,
    ModelSettings,
    RunConfig,
    Runner,
    TResponseInputItem,
)

## Cell 5: Code-Driven Orchestration Recap and Reframe

Every hands-on lecture so far in this section has let the **model** decide what happens next. A handoff triggers when the model reads the conversation and decides another agent is a better fit. That is **LLM-driven orchestration**: flexible, but the path a run takes can vary from one input to the next.

There is a second style, and it is the subject of this lecture: **code-driven orchestration**. Here, plain Python decides what happens next, not the model. The official SDK documentation puts it this way:

> "Orchestrating via code makes tasks more deterministic and predictable, in terms of speed, cost and performance."

The docs describe four common code-driven patterns:

1. **Structured outputs as a routing signal.** Ask an agent to classify a task into a typed category, then let your code pick the next agent based on that category.
2. **Chaining agents.** Feed the output of one agent directly into the next as input, building a multi-step pipeline like outline → story.
3. **A while loop with an evaluator.** Run a task-performing agent in a loop with a second agent that grades its output, repeating until the evaluator's grade passes.
4. **Parallel execution with `asyncio.gather`.** Run several independent agents at once for speed.

This lecture builds patterns 1 through 3, using a story-outline pipeline as the running example. Pattern 4, parallelisation, is the focus of Lecture 5.7.

## Cell 6: Sequential Chain, Step 1, Generate an Outline

This cell defines the first three agents of the pipeline and runs the first step.

| Agent | Role | `output_type` |
|---|---|---|
| `story_outline_agent` | Writes a short story outline from the user's prompt | none (plain `str`) |
| `outline_checker_agent` | Reads an outline and judges its quality and genre | `OutlineCheckerOutput` |
| `story_agent` | Writes the full short story from an approved outline | none (plain `str`) |

`OutlineCheckerOutput` is a Pydantic model with two boolean fields, `good_quality` and `is_scifi`. This is the "structured outputs as a routing signal" pattern from the recap above: because the checker's output is typed, your code can inspect `check.good_quality` directly in an `if` statement a few cells from now, instead of trying to parse free-form text.

All three agents use `ModelSettings(reasoning=Reasoning(effort="none"), verbosity="low")`. None of these tasks benefit from extended reasoning, so keeping effort at `"none"` keeps the pipeline fast and cheap.

The `Runner.run()` call at the bottom passes a `run_config` with `workflow_name="Deterministic story flow"` and `group_id="story-run-001"`. Every step in this pipeline will reuse the same `group_id`, which is what links them together as a single flow in the traces dashboard, even though each step is an independent `Runner.run()` call rather than a continuing conversation.

In [5]:
story_outline_agent = Agent(
    name="Story Outline Agent",
    instructions=(
        "Generate a very short story outline based on "
        "the user's input."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)


class OutlineCheckerOutput(BaseModel):
    good_quality: bool
    is_scifi: bool


outline_checker_agent = Agent(
    name="Outline Checker Agent",
    instructions=(
        "Read the given story outline, and judge the "
        "quality. Also, determine if it is a scifi story."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=OutlineCheckerOutput,
)


story_agent = Agent(
    name="Story Agent",
    instructions="Write a short story based on the given outline.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

outline_result = await Runner.run(
    story_outline_agent,
    "Write a short sci-fi story about a lighthouse keeper.",
    run_config=RunConfig(
        workflow_name="Deterministic story flow",
        group_id="story-run-001",
    ),
)
print("Outline generated:")
print(outline_result.final_output)

Outline generated:
A solitary keeper tends a lighthouse on a storm-wracked moon, its beam guiding not ships but slow-falling cargo pods through the clouds. One night, the light begins flashing a pattern he’s never seen—an emergency code from deep space. Following it, he discovers the lighthouse is actually a buried alien beacon, and the “sea” below is a living atmospheric ocean. As the moon’s crust starts to crack, he must decide whether to keep the light burning to save the colony… or answer the beacon and reveal humanity’s first contact.


## Cell 7: The Quality Gate, Check Before Proceeding

This cell feeds the outline from Cell 6 into `outline_checker_agent`, then uses the typed result to decide whether the pipeline should continue.

Notice that `outline_result.final_output` (a plain string from `story_outline_agent`) is passed directly as the `input` to the checker agent's `Runner.run()` call. That is the "chaining agents" pattern from the recap: one agent's output becomes the next agent's input, with no SDK machinery in between.

`checker_result.final_output` is a real `OutlineCheckerOutput` instance because `outline_checker_agent` has `output_type=OutlineCheckerOutput`. That means `check.good_quality` and `check.is_scifi` are real Python booleans your code can branch on directly, with no parsing and no ambiguity.

The `if` / `elif` / `else` gate below is ordinary Python. No agent decides whether to proceed. Your code does, based on two typed fields.

In [6]:
checker_result = await Runner.run(
    outline_checker_agent,
    outline_result.final_output,
    run_config=RunConfig(
        workflow_name="Deterministic story flow",
        group_id="story-run-001",
    ),
)
check: OutlineCheckerOutput = checker_result.final_output
print(f"Good quality: {check.good_quality}")
print(f"Is sci-fi: {check.is_scifi}")

if not check.good_quality:
    print("Outline is not good quality. Stopping pipeline.")
elif not check.is_scifi:
    print("Outline is not sci-fi. Stopping pipeline.")
else:
    print(
        "Outline passed both checks. Continuing to "
        "write the story."
    )

Good quality: True
Is sci-fi: True
Outline passed both checks. Continuing to write the story.


## Cell 8: Completing the Pipeline, Output Feeds Into the Next Agent

If the gate in Cell 7 passed, this cell runs the final step: `story_agent` writes the full story from the approved outline.

`outline_result.final_output`, the same string produced back in Cell 6, becomes the literal `input` to `story_agent`. There is no `to_input_list()` call here, because these three steps are independent single-turn calls rather than one continuing conversation. Each step gets exactly the input it needs and nothing more. You'll see a genuinely continuing conversation, where `to_input_list()` matters, in Cell 9.

In [7]:
if check.good_quality and check.is_scifi:
    story_result = await Runner.run(
        story_agent,
        outline_result.final_output,
        run_config=RunConfig(
            workflow_name="Deterministic story flow",
            group_id="story-run-001",
        ),
    )
    print("Final story:")
    print(story_result.final_output)

Final story:
The lighthouse had no right to stand where it did.

It rose from a black cliff of basalt on the wind-ripped moon of Pelion, its white tower half-buried in centuries of salt-gray ash. Below it, the “sea” rolled in endless storms—clouds stacked and churning far beneath the ledge, lit from within by blue lightning. Cargo pods drifted through that atmosphere on slow descent paths, their hulls blinking as they followed the beam.

Arin was the keeper.

He had been alone on Pelion for eleven years, long enough to know the groan of every hinge in the lamp room, the temper of every relay, the way the stormfronts changed taste in the air recyclers before they struck. His life was a ritual: trim the lens, feed the furnace, polish the brass, keep the light steady.

The colony below depended on it.

Without the beam, the pods would vanish into the clouds and break on the mountain teeth hidden beneath them. Without the pods, there would be no food, no tools, no oxygen filters. The colon

## Cell 9: The While-Loop-With-Evaluator Pattern

This cell builds pattern 3 from the recap: a task-performing agent running in a loop with a second agent that grades its output, repeating until the grade passes.

| Agent | Role | `output_type` |
|---|---|---|
| `story_generator` | Writes an outline, and revises it if given feedback | none (plain `str`) |
| `evaluator` | Grades an outline as `"pass"`, `"needs_improvement"`, or `"fail"`, with feedback | `EvaluationFeedback` |

`EvaluationFeedback` is a `@dataclass` rather than a Pydantic `BaseModel`. Both work as `output_type` values in the SDK; a dataclass is a convenient choice here since `feedback` and `score` are simple flat fields with no validation logic needed.

Unlike Cells 6 through 8, this **is** a continuing conversation. `input_items` starts as a one-item list and grows every round: `story_outline_result.to_input_list()` converts the run into a replayable list of input items that includes everything so far, and the feedback message appended at the bottom of the loop becomes part of that history on the next round. That's why `to_input_list()` matters here in a way it didn't for the independent single-turn calls in Cells 6 to 8.

`ItemHelpers.text_message_outputs(outline_result.new_items)` pulls the plain text out of the generator's message items so you have a readable `latest_outline` string to print at the end, separate from the input-item history.

Watch the `max_rounds` cap closely. An LLM evaluator can be stubbornly hard to satisfy, so this loop hard-stops after 3 rounds even if the evaluator never returns `"pass"`. Never ship an evaluator loop without a cap like this one; an unbounded `while True` with no exit condition other than the evaluator's opinion is a real risk of an infinite loop in production.

Run the cell and watch the score printed each round. That's the deterministic stopping condition doing its job, one round at a time.

In [8]:
@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["pass", "needs_improvement", "fail"]


story_generator = Agent(
    name="Story Generator",
    instructions=(
        "You generate a very short story outline based "
        "on the user's input. "
        "If there is any feedback provided, use it to "
        "improve the outline."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

evaluator = Agent(
    name="Evaluator",
    instructions=(
        "You evaluate a story outline and decide if it's "
        "good enough. If it's not good enough, provide "
        "feedback on what needs improving."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=EvaluationFeedback,
)

input_items: list[TResponseInputItem] = [
    {"role": "user", "content": "A detective story set on Mars."}
]
max_rounds = 3
rounds = 0

while True:
    outline_result = await Runner.run(story_generator, input_items)
    input_items = outline_result.to_input_list()
    latest_outline = ItemHelpers.text_message_outputs(
        outline_result.new_items
    )
    print(f"Round {rounds + 1}: outline generated")

    eval_result = await Runner.run(evaluator, input_items)
    feedback: EvaluationFeedback = eval_result.final_output
    print(f"  Evaluator score: {feedback.score}")

    if feedback.score == "pass":
        print("  Outline approved.")
        break

    rounds += 1
    if rounds >= max_rounds:
        print("  Max rounds reached. Stopping.")
        break

    input_items.append({
        "role": "user",
        "content": f"Feedback: {feedback.feedback}",
    })

print("\nFinal outline:")
print(latest_outline)

Round 1: outline generated
  Evaluator score: needs_improvement
Round 2: outline generated
  Evaluator score: pass
  Outline approved.

Final outline:
- **Protagonist:** Detective Lena Voss, a burned-out former Earth homicide cop now working security for Valles Outpost, a struggling Mars mining colony.  
- **Stakes:** If she doesn’t solve the case fast, the colony’s water rationing will collapse, a corporate takeover will follow, and Lena’s own freedom depends on proving she wasn’t involved.  
- **Mystery:** A senior hydro-tech is found dead inside a sealed maintenance tunnel with no suit breach, but the tunnel’s airlock logs show a 12-minute gap that shouldn’t exist.  

- **Suspects:**  
  - **Colony Director Harrow:** Wants to hide the water shortage and keep investors calm.  
  - **Union boss Imani Rios:** Openly fighting the corporation over labor and ration cuts.  
  - **Dr. Sato, terraforming engineer:** Knows the hidden aquifer maps and may have altered them.  
  - **Dead man’s 

## Cell 10: Comparing to LLM-Driven Equivalents

| Code-driven pattern (this lecture) | LLM-driven equivalent (5.2–5.5) |
|---|---|
| `if check.good_quality:` gate | Handoff decided by the model reading intent |
| Sequential `Runner.run()` calls | Handoff chain where the model decides the next agent |
| `while` loop with evaluator `output_type` | Agent looping via its own tool-use behaviour |
| Deterministic, same path every time | Emergent, path varies by input |
| Fully unit-testable, mock each `Runner.run()` | Harder to test, depends on model behaviour |

Neither style is strictly better. LLM-driven orchestration, the handoffs and triage patterns from 5.2 through 5.5, shines when the right next step genuinely depends on interpreting open-ended user intent. Code-driven orchestration shines when the steps are fixed and the only question is whether each one succeeded.

## Cell 11: When Code-Driven Pipelines Shine

Reach for the patterns in this notebook when:

- **Content generation pipelines have fixed stages.** Outline, draft, critique, polish, and similar multi-stage workflows map naturally onto a sequential chain.
- **A workflow needs a hard quality gate before proceeding.** If a step must pass a check before the next (expensive) step runs, a typed `output_type` and a plain `if` statement is the simplest way to enforce that.
- **Cost control matters.** A cheap classification or checking step, like `outline_checker_agent` above, can avoid running an expensive agent on input that was never going to pass anyway.
- **Auditability matters.** Compliance-sensitive pipelines benefit from every step being traceable and reproducible in plain code, not emergent from a model's reasoning.
- **Testing matters.** Each step in a code-driven pipeline can be unit tested independently with mocked agent outputs, since the control flow lives in your code rather than inside the model.

Lecture 5.7 picks up pattern 4 from Cell 5's recap: running multiple agents at once with `asyncio.gather`, for cases where several independent steps don't need to wait on each other at all.